<a href="https://colab.research.google.com/github/EgorMehed/ML-homework/blob/main/%D0%9B%D0%A0_9_%D0%A0%D0%B5%D0%BA%D0%BE%D0%BC%D0%B5%D0%BD%D0%B4%D0%B0%D1%86%D1%96%D0%B9%D0%BD%D1%96_%D1%81%D0%B8%D1%81%D1%82%D0%B5%D0%BC%D0%B8_%D0%9C%D0%B5%D1%85%D0%B5%D0%B4_%D0%84%D0%B3%D0%BE%D1%80.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Мехед Єгор  
**Група:** ФІТ3-12  
**Дата:** 24.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [1]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [2]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [3]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [4]:
# TODO 2.1 — Дослідіть набір даних:
# - кількість унікальних користувачів;
# - кількість унікальних фільмів;
# - мінімальне, максимальне, середнє значення rating;
# - пропущені значення (за наявності).

# Ваш код:
n_users = ratings_df['user_id'].nunique()
n_items = ratings_df['item_id'].nunique()
min_rating = ratings_df['rating'].min()
max_rating = ratings_df['rating'].max()
mean_rating = ratings_df['rating'].mean()

print(f'Кількість унікальних користувачів: {n_users}')
print(f'Кількість унікальних фільмів: {n_items}')
print(f'Мінімальна оцінка: {min_rating}')
print(f'Максимальна оцінка: {max_rating}')
print(f'Середня оцінка: {mean_rating:.4f}')

print('\nРозподіл оцінок:')
print(ratings_df['rating'].value_counts().sort_index())

print('\nПропущені значення за колонками:')
print(ratings_df.isna().sum())

Кількість унікальних користувачів: 943
Кількість унікальних фільмів: 1682
Мінімальна оцінка: 1.0
Максимальна оцінка: 5.0
Середня оцінка: 3.5299

Розподіл оцінок:
rating
1.0     6110
2.0    11370
3.0    27145
4.0    34174
5.0    21201
Name: count, dtype: int64

Пропущені значення за колонками:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64


**Короткі спостереження щодо даних:**

Набір даних `MovieLens 100K` містить 100 000 оцінок від 943 унікальних користувачів для 1 682 унікальних фільмів, при цьому пропущених значень у жодній колонці немає. Оцінки задані за цілочисловою шкалою від 1.0 до 5.0 із середнім значенням 3.5299, що свідчить про помірне зміщення розподілу у бік позитивних оцінок (найчастіше трапляються оцінки 4 та 3). Матриця взаємодій є розрідженою (заповнено близько 6.3% усіх можливих пар «користувач–фільм»), що є типовою умовою для задач колаборативної фільтрації.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [5]:
# TODO 3.1 — Створіть train_df та test_df із ratings_df.
# Використайте sklearn_train_test_split(..., test_size=0.2, random_state=42).

# Ваш код:
train_df, test_df = sklearn_train_test_split(
    ratings_df, test_size=0.2, random_state=42
)

print('Розмір train_df:', train_df.shape)
print('Розмір test_df:', test_df.shape)

Розмір train_df: (80000, 4)
Розмір test_df: (20000, 4)


In [6]:
# TODO 3.2 — Підготуйте об’єкти для Surprise:
# 1. reader = Reader(rating_scale=...)
#    Підказка: data.reader.rating_scale
# 2. train_data = Dataset.load_from_df(...)
#    Візьміть тільки user_id, item_id, rating із train_df.
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
# 4. Перевірте розміри train/test.

# Ваш код:
reader = Reader(rating_scale=data.reader.rating_scale)
train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']],
    reader
)
testset = list(
    test_df[['user_id', 'item_id', 'rating']].itertuples(index=False, name=None)
)

print('Шкала оцінок:', data.reader.rating_scale)
print('Кількість записів у train_data:', len(train_data.raw_ratings))
print('Кількість записів у testset:', len(testset))
print('Приклад запису testset:', testset[0])

Шкала оцінок: (1, 5)
Кількість записів у train_data: 80000
Кількість записів у testset: 20000
Приклад запису testset: ('877', '381', 4.0)


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [7]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

Для `SVD` залишено базову сітку, що охоплює ключові фактори якості матричної факторизації: `n_factors` (`[50, 100]`) визначає розмірність латентного простору (баланс між виразністю моделі та перенавчанням), `lr_all` (`[0.005, 0.010]`) контролює крок стохастичного градієнтного спуску, а `reg_all` (`[0.02, 0.05]`) задає силу L2-регуляризації для зменшення перенавчання на розріджених даних.

Для `KNNBasic` сітка перевіряє кількість найближчих сусідів `k` (`[20, 40]`), поріг мінімальної кількості сусідів `min_k` (`[1, 5]`), метрику схожості `name` (`cosine` проти `pearson`, де Пірсон додатково враховує індивідуальний зсув середніх оцінок користувачів) та напрямок фільтрації `user_based` (`False` — item-based, `True` — user-based). Такий набір дозволяє коректно порівняти основні режими роботи обох алгоритмів без надмірних обчислювальних витрат.

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [8]:
# TODO 5.1 — Підберіть гіперпараметри SVD.
# Підказка: GridSearchCV(SVD, param_grid_svd,
#                        measures=['rmse', 'mae'], cv=3, n_jobs=-1)
# Після створення викличте .fit(train_data).

# Ваш код:
grid_search_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)
grid_search_svd.fit(train_data)

In [9]:
# TODO 5.2 — Підберіть гіперпараметри KNNBasic.
# Підказка: використайте param_grid_knn та ті самі метрики/cv.

# Ваш код:
grid_search_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)
grid_search_knn.fit(train_data)

In [10]:
# TODO 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
# Підказки:
# grid_search_svd.best_score['rmse']
# grid_search_svd.best_params['rmse']
# grid_search_knn.best_score['rmse']

# Ваш код:
print('=== SVD (GridSearchCV) ===')
print(f"Найкращий CV RMSE: {grid_search_svd.best_score['rmse']:.4f}")
print(f"Найкращий CV MAE:  {grid_search_svd.best_score['mae']:.4f}")
print('Найкращі параметри за RMSE:', grid_search_svd.best_params['rmse'])

print('\n=== KNNBasic (GridSearchCV) ===')
print(f"Найкращий CV RMSE: {grid_search_knn.best_score['rmse']:.4f}")
print(f"Найкращий CV MAE:  {grid_search_knn.best_score['mae']:.4f}")
print('Найкращі параметри за RMSE:', grid_search_knn.best_params['rmse'])

=== SVD (GridSearchCV) ===
Найкращий CV RMSE: 0.9412
Найкращий CV MAE:  0.7435
Найкращі параметри за RMSE: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}

=== KNNBasic (GridSearchCV) ===
Найкращий CV RMSE: 1.0279
Найкращий CV MAE:  0.8148
Найкращі параметри за RMSE: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | *(див. вивід комірки 5.3)* | *(див. вивід комірки 5.3)* | *(див. вивід комірки 5.3)* |
| KNNBasic | *(див. вивід комірки 5.3)* | *(див. вивід комірки 5.3)* | *(див. вивід комірки 5.3)* |

За результатами 3-кратної кросвалідації на навчальній вибірці модель `SVD` стабільно демонструє нижчі значення як `RMSE`, так і `MAE` порівняно з `KNNBasic`. Це пояснюється тим, що матрична факторизація узагальнює приховані вподобання користувачів та характеристики фільмів у латентному просторі, тоді як `KNNBasic` покладається лише на пряму схожість за спільними оцінками, що є менш стійким до розрідженості матриці. Найкращі параметри для обох моделей визначаються автоматично за мінімальним CV RMSE і передаються на етап фінального навчання.

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [11]:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
# TODO 6.2 — Створіть trainset із train_data.
# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.

# Ваш код:
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']

trainset = train_data.build_full_trainset()

best_svd = SVD(**best_svd_params, random_state=42)
best_svd.fit(trainset)

best_knn = KNNBasic(**best_knn_params)
best_knn.fit(trainset)

print('Моделі best_svd та best_knn успішно навчено на повній навчальній вибірці.')
print('Кількість користувачів у trainset:', trainset.n_users)
print('Кількість фільмів у trainset:', trainset.n_items)

Computing the cosine similarity matrix...
Done computing similarity matrix.
Моделі best_svd та best_knn успішно навчено на повній навчальній вибірці.
Кількість користувачів у trainset: 943
Кількість фільмів у trainset: 1653


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [12]:
# TODO 7.1 — Отримайте svd_predictions та knn_predictions.
# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
# Підказка: accuracy.rmse(predictions, verbose=False)
#           accuracy.mae(predictions, verbose=False)

# Ваш код:
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)

knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

print(f'SVD      -> Test RMSE: {svd_rmse:.4f}, Test MAE: {svd_mae:.4f}')
print(f'KNNBasic -> Test RMSE: {knn_rmse:.4f}, Test MAE: {knn_mae:.4f}')

SVD      -> Test RMSE: 0.9192, Test MAE: 0.7234
KNNBasic -> Test RMSE: 1.0120, Test MAE: 0.7984


In [13]:
# TODO 7.3 — Сформуйте DataFrame results із колонками:
# Model, RMSE, MAE.
# Відсортуйте за RMSE та покажіть порівняння.

# Ваш код:
results = pd.DataFrame([
    {'Model': 'SVD', 'RMSE': svd_rmse, 'MAE': svd_mae},
    {'Model': 'KNNBasic', 'RMSE': knn_rmse, 'MAE': knn_mae}
]).sort_values(by='RMSE').reset_index(drop=True)

display(results)

,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011953,0.798421


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | *(див. таблицю `results` вище)* | *(див. таблицю `results` вище)* |
| KNNBasic | *(див. таблицю `results` вище)* | *(див. таблицю `results` вище)* |

**Порівняння з CV:** Значення `RMSE` та `MAE` на незалежній тестовій вибірці узгоджуються з оцінками кросвалідації (часто є трохи кращими завдяки навчанню на 100% навчальної вибірки замість 2/3 у 3-fold CV). Відсутність різкого погіршення метрик на тесті підтверджує, що моделі не перенавчилися під час підбору гіперпараметрів, а вибір кращої моделі за CV є коректним.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [14]:
# Початковий ID можна змінити
user_id = '196'

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
# TODO 8.2 — Перевірте, що користувач присутній у trainset.
# TODO 8.3 — Знайдіть неоцінені ним фільми.
# TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
# Бажані колонки результату: item_id, predicted_rating.
# Передбачте ситуацію, коли обраного user_id немає у trainset.

# Ваш код:
if grid_search_svd.best_score['rmse'] <= grid_search_knn.best_score['rmse']:
    best_model = best_svd
    best_model_name = 'SVD'
else:
    best_model = best_knn
    best_model_name = 'KNNBasic'

print(f'Обрана модель за CV RMSE: {best_model_name}')

try:
    inner_uid = trainset.to_inner_uid(user_id)
except ValueError:
    print(f"Користувача '{user_id}' не знайдено у навчальній вибірці (trainset). Неможливо сформувати персоналізовані рекомендації.")
else:
    rated_inner_iids = {inner_iid for (inner_iid, _) in trainset.ur[inner_uid]}
    unrated_raw_iids = [
        trainset.to_raw_iid(inner_iid)
        for inner_iid in trainset.all_items()
        if inner_iid not in rated_inner_iids
    ]

    predictions_list = [
        (raw_iid, best_model.predict(uid=user_id, iid=raw_iid).est)
        for raw_iid in unrated_raw_iids
    ]

    top_10_recommendations = (
        pd.DataFrame(predictions_list, columns=['item_id', 'predicted_rating'])
        .sort_values(by='predicted_rating', ascending=False)
        .head(10)
        .reset_index(drop=True)
    )

    print(f"Топ-10 рекомендацій для користувача '{user_id}' (модель {best_model_name}):")
    display(top_10_recommendations)

Обрана модель за CV RMSE: SVD
Топ-10 рекомендацій для користувача '196' (модель SVD):


,item_id,predicted_rating
0,320,4.490358
1,64,4.470680
2,480,4.413710
3,313,4.401920
4,513,4.399624
5,483,4.398092
6,318,4.392102
7,656,4.385379
8,22,4.377388
9,272,4.376010


## 9. Перевірка виконання

Позначте після реалізації:

- [x] Дані `ml-100k` завантажено та перевірено.
- [x] Є розбиття 80/20 з фіксованим `random_state`.
- [x] `Reader`, `train_data` і `testset` створені коректно.
- [x] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [x] Для SVD і KNNBasic показано параметри та результати CV.
- [x] Моделі навчено на повній навчальній частині.
- [x] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [x] Наведено порівняльну таблицю та інтерпретацію.
- [x] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [x] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. **Яку модель обрано за кросвалідацією і чому?** За результатами 3-кратної кросвалідації за метрикою `RMSE` обрано модель `SVD`, оскільки вона забезпечує меншу похибку прогнозування оцінок порівняно з `KNNBasic` завдяки використанню латентних факторів та регуляризації.
2. **Які найкращі гіперпараметри отримано для обох моделей?** Оптимальні комбінації параметрів `best_svd_params` (з параметрами `n_factors`, `lr_all`, `reg_all`) та `best_knn_params` (`k`, `min_k`, `sim_options`) визначено автоматично за допомогою `GridSearchCV` (виведено у п. 5.3).
3. **Які підсумкові значення RMSE та MAE на тестовій вибірці?** Підсумкові значення `Test RMSE` та `Test MAE` для обох моделей розраховано на незалежному `testset` і зведено у відсортованій таблиці `results` (п. 7.3), де `SVD` посідає перше місце.
4. **Чи є різниця між результатами CV та тестування?** Показники на тестовій вибірці близькі до результатів кросвалідації (і зазвичай дещо кращі за рахунок більшого обсягу навчальних даних при фінальному навчанні), що підтверджує хорошу узагальнювальну здатність моделей та відсутність перенавчання.
5. **Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?** Основним обмеженням є проблема «холодного старту» (cold start): для нових користувачів або нових фільмів, яких не було у навчальній вибірці, модель не може побудувати персоналізовані латентні вектори чи знайти сусідів і змушена покладатися на глобальне середнє. Крім того, моделі не використовують контентні ознаки (жанри, акторів, рік виходу) та часову динаміку вподобань.

**Висновок:** У ході лабораторної роботи було досліджено набір даних `MovieLens 100K` та розділено його на навчальну і тестову вибірки без витоку даних. За допомогою `GridSearchCV` на навчальній частині підібрано оптимальні гіперпараметри для алгоритмів `SVD` та `KNNBasic`, після чого обидві моделі було перенавчено на повній навчальній вибірці. Оцінювання на незалежній тестовій вибірці підтвердило перевагу матричної факторизації `SVD` над методом найближчих сусідів `KNNBasic` за метриками `RMSE` та `MAE`. Розбіжність між результатами кросвалідації та тестування виявилася мінімальною. На основі кращої за CV моделі було сформовано список топ-10 персоналізованих рекомендацій для користувача `'196'` із виключенням уже оцінених ним фільмів та обробкою випадку відсутності користувача в `trainset`. Додатково реалізовано власний алгоритм матричної факторизації (`CustomBiasedSVD`) з нуля на `NumPy`, що дозволяє глибше зрозуміти механіку стохастичного градієнтного спуску в рекомендаційних системах.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**

In [15]:
# Власна реалізація алгоритму рекомендацій з нуля: Biased SVD (матрична факторизація через SGD) на NumPy
class CustomBiasedSVD:
    """
    Прогноз оцінки: r_hat(u, i) = mu + b_u + b_i + dot(P[u], Q[i])
    Навчання виконується методом стохастичного градієнтного спуску (SGD) з L2-регуляризацією.
    """
    def __init__(self, n_factors=50, lr=0.005, reg=0.05, n_epochs=15, rating_scale=(1.0, 5.0), random_state=42):
        self.n_factors = n_factors
        self.lr = lr
        self.reg = reg
        self.n_epochs = n_epochs
        self.min_rating, self.max_rating = rating_scale
        self.random_state = random_state

    def fit(self, train_df):
        rng = np.random.default_rng(self.random_state)

        unique_users = train_df['user_id'].unique()
        unique_items = train_df['item_id'].unique()

        self.user_to_idx = {u: idx for idx, u in enumerate(unique_users)}
        self.item_to_idx = {i: idx for idx, i in enumerate(unique_items)}
        self.idx_to_item = {idx: i for i, idx in self.item_to_idx.items()}

        n_users = len(unique_users)
        n_items = len(unique_items)

        self.global_mean = float(train_df['rating'].mean())
        self.user_bias = np.zeros(n_users, dtype=np.float64)
        self.item_bias = np.zeros(n_items, dtype=np.float64)
        self.P = rng.normal(0.0, 0.1, size=(n_users, self.n_factors))
        self.Q = rng.normal(0.0, 0.1, size=(n_items, self.n_factors))

        u_indices = train_df['user_id'].map(self.user_to_idx).to_numpy()
        i_indices = train_df['item_id'].map(self.item_to_idx).to_numpy()
        r_values = train_df['rating'].to_numpy(dtype=np.float64)

        self.user_rated_items = train_df.groupby('user_id')['item_id'].apply(set).to_dict()

        n_samples = len(r_values)
        order = np.arange(n_samples)

        for epoch in range(self.n_epochs):
            rng.shuffle(order)
            for u, i, r in zip(u_indices[order].tolist(), i_indices[order].tolist(), r_values[order].tolist()):
                pred = self.global_mean + self.user_bias[u] + self.item_bias[i] + float(np.dot(self.P[u], self.Q[i]))
                err = r - pred

                self.user_bias[u] += self.lr * (err - self.reg * self.user_bias[u])
                self.item_bias[i] += self.lr * (err - self.reg * self.item_bias[i])

                p_u = self.P[u].copy()
                self.P[u] += self.lr * (err * self.Q[i] - self.reg * self.P[u])
                self.Q[i] += self.lr * (err * p_u - self.reg * self.Q[i])

        return self

    def predict(self, user_id, item_id):
        pred = self.global_mean
        u_known = user_id in self.user_to_idx
        i_known = item_id in self.item_to_idx

        if u_known:
            u = self.user_to_idx[user_id]
            pred += self.user_bias[u]
        if i_known:
            i = self.item_to_idx[item_id]
            pred += self.item_bias[i]
        if u_known and i_known:
            pred += float(np.dot(self.P[u], self.Q[i]))

        return float(np.clip(pred, self.min_rating, self.max_rating))

    def evaluate(self, test_df):
        preds = np.array([
            self.predict(u, i)
            for u, i in zip(test_df['user_id'], test_df['item_id'])
        ])
        actuals = test_df['rating'].to_numpy(dtype=np.float64)
        rmse = float(np.sqrt(np.mean((actuals - preds) ** 2)))
        mae = float(np.mean(np.abs(actuals - preds)))
        return rmse, mae

    def recommend_top_n(self, user_id, n=10):
        if user_id not in self.user_to_idx:
            print(f"Користувача '{user_id}' немає у навчальній вибірці.")
            return pd.DataFrame(columns=['item_id', 'predicted_rating'])

        u = self.user_to_idx[user_id]
        rated = self.user_rated_items.get(user_id, set())

        all_preds = self.global_mean + self.user_bias[u] + self.item_bias + self.Q @ self.P[u]
        all_preds = np.clip(all_preds, self.min_rating, self.max_rating)

        candidates = [
            (self.idx_to_item[i_idx], float(all_preds[i_idx]))
            for i_idx in range(len(self.idx_to_item))
            if self.idx_to_item[i_idx] not in rated
        ]

        return (
            pd.DataFrame(candidates, columns=['item_id', 'predicted_rating'])
            .sort_values(by='predicted_rating', ascending=False)
            .head(n)
            .reset_index(drop=True)
        )

# Навчання та оцінювання власного алгоритму на тих самих train_df / test_df
custom_svd = CustomBiasedSVD(
    n_factors=50,
    lr=0.005,
    reg=0.05,
    n_epochs=15,
    rating_scale=data.reader.rating_scale,
    random_state=42
)
custom_svd.fit(train_df)
custom_rmse, custom_mae = custom_svd.evaluate(test_df)

print(f'CustomBiasedSVD -> Test RMSE: {custom_rmse:.4f}, Test MAE: {custom_mae:.4f}')

# Порівняння власного алгоритму з моделями бібліотеки Surprise
all_results = pd.concat([
    results,
    pd.DataFrame([{'Model': 'CustomBiasedSVD (з нуля)', 'RMSE': custom_rmse, 'MAE': custom_mae}])
], ignore_index=True).sort_values(by='RMSE').reset_index(drop=True)

display(all_results)

# Топ-10 рекомендацій від власної моделі для користувача user_id
print(f"Топ-10 рекомендацій від CustomBiasedSVD для користувача '{user_id}':")
display(custom_svd.recommend_top_n(user_id, n=10))

CustomBiasedSVD -> Test RMSE: 0.9369, Test MAE: 0.7405


,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,CustomBiasedSVD (з нуля),0.936861,0.740506
2,KNNBasic,1.011953,0.798421


Топ-10 рекомендацій від CustomBiasedSVD для користувача '196':


,item_id,predicted_rating
0,318,4.445478
1,114,4.422995
2,64,4.382231
3,483,4.379814
4,513,4.354365
5,50,4.349302
6,408,4.342972
7,169,4.329108
8,603,4.294056
9,479,4.279713
